In [41]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle
from sklearn.metrics import roc_auc_score


In [42]:
df_saudi_tree = pd.read_excel("saudi_dataset/Employee attrition dataset for tree-based models.xlsx")
df_saudi_nontree = pd.read_excel("saudi_dataset/Employee attrition dataset for non-tree-based models.xlsx")

In [43]:
df_saudi_tree.head()

,Attrition,Gender,Age,Maritalstatus,Academic_degree,Years_Experience,Years_experience_lastorganization,Sector,Department,JobTitle,...,Job_Engagement,Distance_to_work,Work_Live_Balance,Physical_Stress,Psychological_Exhaustion,Job_Stability,Health_Issues,Environment_Satisfaction,Job_Satisfaction,Job_Opportunities
0,1,0,1,0.405953,2,0,0,0.359278,0.600378,0.623277,...,1,1,1,2,0,0,1,1,0,1
1,0,0,0,0.447209,2,0,0,0.359278,0.600378,0.623277,...,1,1,1,0,0,1,0,1,1,0
2,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,0,1,2,1,0,0,1,1,1
3,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,1,1,2,2,0,0,0,0,0
4,0,0,0,0.447209,1,0,0,0.260830,0.229320,0.190450,...,1,1,1,1,1,0,0,1,0,0


In [44]:
columns_to_delete = ["Maritalstatus", "Department", "JobTitle", "Allowances"]
# allowances and marital status dropped since legend not clear 
# department and jobtitle dropped since we predict attrition for each sector, want enough data so we aggregate over the departments/jobs

df_saudi_tree = df_saudi_tree.drop(columns=columns_to_delete)

In [45]:
# Convert categorical string codes to integers
# 0.565187=1, 0.359278=2, 0.260830=3, 0.456738=4, 0.512493=5
sector_map = {0.5651866011647716: 1, 0.3592782301026135: 2, 0.2608303362988201: 3, 0.4567380973729042: 4, 0.5124929017603634: 5}

df_saudi_tree['Sector'] = df_saudi_tree['Sector'].map(sector_map)


In [46]:
df_saudi_tree.shape

(1174, 30)

In [47]:
# define protected attributes 
protected_attributes = ["Gender", "Age"]


In [48]:
random_state = 1234
saudi_train, saudi_test = train_test_split(df_saudi_tree, test_size=0.2, random_state=random_state)

saudi_train.to_parquet("saudi_dataset/train_cleaned.parquet")
saudi_test.to_parquet("saudi_dataset/test_cleaned.parquet")

# Separate features and target
x_train = saudi_train.drop("Attrition", axis=1)
y_train = saudi_train["Attrition"]
x_test = saudi_test.drop("Attrition", axis=1)
y_test = saudi_test["Attrition"]


In [49]:
# random forest model 
rf_model = RandomForestClassifier(n_estimators=100, random_state=random_state, class_weight='balanced')
rf_model.fit(x_train, y_train)

#evaluate the model based on accuracy and AUC metrics 
rf_pred = rf_model.predict(x_test)
target_pred_proba = rf_model.predict_proba(x_test)[:, 1]  # Get probability for class 1
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, target_pred_proba)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")
print(f"AUC: {rf_auc * 100:.2f}%")


# Save the model
with open('saudi_dataset/RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Accuracy: 82.13%
AUC: 88.23%


In [50]:
# Define feature types for saudi dataset - most are categorical (ordinal)
numerical_features = []  # Saudi dataset contains primarily categorical features
ordinal_features = list(x_train.columns)  # All features are ordinal categorical
nominal_features = []

# Define categorical features (ordinal + nominal)
categorical_features = ordinal_features + nominal_features

# Helper function to calculate feature distributions
def get_feature_distribution(feature_name, data, is_categorical):
    """Calculate distribution (value counts/percentages) for categorical features"""
    if not is_categorical:
        return None
    
    value_counts = data[feature_name].value_counts().sort_index()
    total = len(data)
    distribution_str = ""
    for value, count in value_counts.items():
        pct = (count / total) * 100
        distribution_str += f"Value {value}: {pct:.1f}%, "
    
    return distribution_str.rstrip(", ") if distribution_str else None

feature_desc = ['the gender of the employee (0 = female, 1 = male)', 
                'the age of the employee (0 = 21-30, 1 = 31-40, 3= 41 and more)',
                'the highest obtained academic degree of the employee (0 = secondary school, 1 = bachelor, 2 = master, 3 = PhD)',
                'the total years of experience of the employee overall all their jobs (0 = 1-5 years, 1 = 6-10 years, 2 = 11 and more years)',
                'the years of experience the employee has in this job (0 = 1-5 years, 1 = 6-10 years, 2 = 11 and more years)',
                'the sector in which the employee is employed (1 = other , 2 = medical, 3 = education, 4 = financial, 5 = food)',
                'the monthly salary of the employee in SAR (0 = 1k-5k, 1 = 6k-10k, 2 = 11k-15k, 3 = 16k and more)',
                'whether the emloyee has medical ensurance in this job (0 = no, 1 = yes)',
                'whether the employee received an annual bonus for their performance in this job (0 = no, 1 = yes)',
                'whether the employee had overtime in this job (0 = no, 1 = yes)', 
                'whether the employee received compenstation for their possible overtime in this job (0 = no overtime, 1 = no, 2 = yes)',
                'whether the employee is satisfied with their income relative to their effort in this job (0 = no, 1 = yes)',
                'whether the employee felt they got the promotion they deserved in this job (0 = no, 1 = yes)',
                'the amount of training programs the employee attended in the last 3 years of this job (0 = no trainings, 1 = 1-3, 2 = 4-6, 3 = 7 and more trainings)',
                'whether the employee benefited from the training programs provided by their current employer (0 = no, 1 = yes)',
                'how often the employee travels for business purposes in this job (0 = never, 1 = rarely, 2 = frequently)',
                'the level of support the employee receives from their organization in this job (0 = low, 1 = medium, 2 = high)',
                'whether the employee feels feel the moral appreciation and recognition of their effort by their superiors in this job (0 = no, 1 = yes)',
                'the emotional commitment and psychological relationship of the employee with their current organization (0 = low, 1 = medium, 2 = high)', 
                'how easy it was to get involved in the job (e.g. decision making, opinions) of the employee in this job (0 = easy, 1 = medium, 2 = difficult)', 
                'the distance to the employee his/her workplace in this job (0 = close, 1 = medium, 2 = far)',
                'how easy it is to balance work life and personal life at work for the employee in this job (0 = easy, 1 = medium, 2 = difficult)',
                'whether the employee felt physically stressed due to physically demanding tasks in this job (0 = no, 1 = sometimes, 2 = yes)',
                'whether the employee felt a state of psychological exhaustion and mental or emotional fatigue as a result of their constant exposure to job stress in this job (0 = no, 1 = sometimes, 2 = yes)',
                'whether the employee feels a sense of job security and stability in this job (0 = no, 1 = yes)',
                'whether the employee did have any health issues that foreced them to leave this job (0 = no, 1 = yes)',
                'the satisfaction of the employee with their work environment in this job (0 = low, 1 = medium, 2 = high)',
                'the satisfaction of the employee with this job in general (0 = not satisfied, 1 = satisfied, 2 = very satisfied)',
                'whether the employee received other job opportunities while working in this job (0 = no, 1 = yes)'
]

# Build feature dataframe with distributions for categorical features
feature_data = []
for col in x_train.columns:
    is_cat = col in categorical_features
    row = {
        "feature_name": col,
        "feature_average": x_train[col].mean() if not is_cat else None,
        "feature_distribution": get_feature_distribution(col, x_train, is_cat),
        "feature_desc": feature_desc[list(x_train.columns).index(col)]
    }
    feature_data.append(row)

feature_desc_df = pd.DataFrame(feature_data)
     

dataset_description="data about the Saudi private sector to better understand the causes and consequences of employee satisfaction and turnover. The initial step to collect this data was administering an online survey to 1,200 employees of various Saudi Arabian companies. The dataset's question axes were described using a total of 29 qualities which you are given"
target_description="The target variable is whether the employee left their last job (1) or not (0)."
task_description="Predict whether an employee will leave the job (1) or not (0) based on their characteristics."

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('saudi_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [51]:
feature_desc_df

,feature_name,feature_average,feature_distribution,feature_desc
0,Gender,None,"Value 0: 57.5%, Value 1: 42.5%","the gender of the employee (0 = female, 1 = male)"
1,Age,None,"Value 0: 49.5%, Value 1: 38.6%, Value 2: 11.9%","the age of the employee (0 = 21-30, 1 = 31-40,..."
2,Academic_degree,None,"Value 0: 20.4%, Value 1: 65.1%, Value 2: 12.1%...",the highest obtained academic degree of the em...
3,Years_Experience,None,"Value 0: 41.9%, Value 1: 42.6%, Value 2: 15.5%",the total years of experience of the employee ...
4,Years_experience_lastorganization,None,"Value 0: 59.3%, Value 1: 29.7%, Value 2: 11.0%",the years of experience the employee has in th...
5,Sector,None,"Value 1: 29.1%, Value 2: 25.0%, Value 3: 23.0%...",the sector in which the employee is employed (...
6,MonthlySalary,None,"Value 0: 41.4%, Value 1: 41.5%, Value 2: 12.8%...",the monthly salary of the employee in SAR (0 =...
7,MedicalInsurance,None,"Value 0: 29.2%, Value 1: 70.8%",whether the emloyee has medical ensurance in t...
8,Bonus,None,"Value 0: 57.7%, Value 1: 42.3%",whether the employee received an annual bonus ...
9,OverTime,None,"Value 0: 57.3%, Value 1: 42.7%",whether the employee had overtime in this job ...


In [52]:
dataset_info

{'dataset_description': "data about the Saudi private sector to better understand the causes and consequences of employee satisfaction and turnover. The initial step to collect this data was administering an online survey to 1,200 employees of various Saudi Arabian companies. The dataset's question axes were described using a total of 29 qualities which you are given",
 'target_description': 'The target variable is whether the employee left their last job (1) or not (0).',
 'task_description': 'Predict whether an employee will leave the job (1) or not (0) based on their characteristics.',
 'feature_description':                                  feature_name feature_average  \
 0                                      Gender            None   
 1                                         Age            None   
 2                             Academic_degree            None   
 3                            Years_Experience            None   
 4           Years_experience_lastorganization     

In [53]:
feature_desc_df

,feature_name,feature_average,feature_distribution,feature_desc
0,Gender,None,"Value 0: 57.5%, Value 1: 42.5%","the gender of the employee (0 = female, 1 = male)"
1,Age,None,"Value 0: 49.5%, Value 1: 38.6%, Value 2: 11.9%","the age of the employee (0 = 21-30, 1 = 31-40,..."
2,Academic_degree,None,"Value 0: 20.4%, Value 1: 65.1%, Value 2: 12.1%...",the highest obtained academic degree of the em...
3,Years_Experience,None,"Value 0: 41.9%, Value 1: 42.6%, Value 2: 15.5%",the total years of experience of the employee ...
4,Years_experience_lastorganization,None,"Value 0: 59.3%, Value 1: 29.7%, Value 2: 11.0%",the years of experience the employee has in th...
5,Sector,None,"Value 1: 29.1%, Value 2: 25.0%, Value 3: 23.0%...",the sector in which the employee is employed (...
6,MonthlySalary,None,"Value 0: 41.4%, Value 1: 41.5%, Value 2: 12.8%...",the monthly salary of the employee in SAR (0 =...
7,MedicalInsurance,None,"Value 0: 29.2%, Value 1: 70.8%",whether the emloyee has medical ensurance in t...
8,Bonus,None,"Value 0: 57.7%, Value 1: 42.3%",whether the employee received an annual bonus ...
9,OverTime,None,"Value 0: 57.3%, Value 1: 42.7%",whether the employee had overtime in this job ...
